In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score
import pickle

In [2]:
def run_model_training():
    print("Starting Model Training & Evaluation...")
    import pandas as pd
    from sklearn.model_selection import train_test_split, RandomizedSearchCV
    from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
    from xgboost import XGBClassifier
    from sklearn.metrics import classification_report, roc_auc_score
    import pickle
    import scipy.stats as stats
    
    df = pd.read_csv('engineered_data.csv')
    
    # Separate features and target
    X = df.drop(columns=['employee_id', 'high_friction'])
    y = df['high_friction']
    
    # Train-test split (80/20)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    print(f"Training set size: {X_train.shape[0]}")
    print(f"Test set size: {X_test.shape[0]}\n")
    
    models = {}
    
    # Model 1: Random Forest
    print("--- Training Random Forest (Tuned) ---")
    rf_params = {
        'n_estimators': stats.randint(100, 500),
        'max_depth': [None, 10, 20, 30],
        'min_samples_split': stats.randint(2, 10),
        'min_samples_leaf': stats.randint(1, 5)
    }
    rf_search = RandomizedSearchCV(RandomForestClassifier(random_state=42), rf_params, n_iter=20, cv=3, scoring='roc_auc', n_jobs=-1, random_state=42)
    rf_search.fit(X_train, y_train)
    rf_model = rf_search.best_estimator_
    rf_probs = rf_model.predict_proba(X_test)[:, 1]
    rf_auc = roc_auc_score(y_test, rf_probs)
    print("Best RF Params:", rf_search.best_params_)
    print(f"RF ROC-AUC: {rf_auc:.4f}\n")
    models['Random Forest'] = (rf_model, rf_auc)
    
    # Model 2: XGBoost
    print("--- Training XGBoost (Tuned) ---")
    xgb_params = {
        'n_estimators': stats.randint(100, 500),
        'max_depth': stats.randint(3, 10),
        'learning_rate': stats.uniform(0.01, 0.2),
        'subsample': stats.uniform(0.6, 0.4),
        'colsample_bytree': stats.uniform(0.6, 0.4)
    }
    xgb_search = RandomizedSearchCV(XGBClassifier(random_state=42, eval_metric='logloss'), xgb_params, n_iter=20, cv=3, scoring='roc_auc', n_jobs=-1, random_state=42)
    xgb_search.fit(X_train, y_train)
    xgb_model = xgb_search.best_estimator_
    xgb_probs = xgb_model.predict_proba(X_test)[:, 1]
    xgb_auc = roc_auc_score(y_test, xgb_probs)
    print("Best XGB Params:", xgb_search.best_params_)
    print(f"XGB ROC-AUC: {xgb_auc:.4f}\n")
    models['XGBoost'] = (xgb_model, xgb_auc)

    # Model 3: HistGradientBoosting
    print("--- Training HistGradientBoosting (Tuned) ---")
    hgb_params = {
        'max_iter': stats.randint(100, 500),
        'max_depth': [None, 10, 20, 30],
        'learning_rate': stats.uniform(0.01, 0.2),
        'min_samples_leaf': stats.randint(20, 50)
    }
    hgb_search = RandomizedSearchCV(HistGradientBoostingClassifier(random_state=42), hgb_params, n_iter=20, cv=3, scoring='roc_auc', n_jobs=-1, random_state=42)
    hgb_search.fit(X_train, y_train)
    hgb_model = hgb_search.best_estimator_
    hgb_probs = hgb_model.predict_proba(X_test)[:, 1]
    hgb_auc = roc_auc_score(y_test, hgb_probs)
    print("Best HGB Params:", hgb_search.best_params_)
    print(f"HGB ROC-AUC: {hgb_auc:.4f}\n")
    models['HistGradientBoosting'] = (hgb_model, hgb_auc)
    
    # Select best model
    best_name = max(models, key=lambda k: models[k][1])
    best_model, best_auc = models[best_name]
    print(f"Selecting {best_name} as the final model with ROC-AUC {best_auc:.4f}.")
    
    print(classification_report(y_test, best_model.predict(X_test)))
    
    # Save the model
    with open('best_model.pkl', 'wb') as f:
        pickle.dump(best_model, f)
    print("Saved 'best_model.pkl'.")
    
    # Save test data for SHAP interpretation later
    X_test.to_csv('X_test.csv', index=False)


In [3]:
if __name__ == '__main__':
    run_model_training()

Starting Model Training & Evaluation...
Training set size: 1176
Test set size: 294

--- Training Random Forest (Tuned) ---


Best RF Params: {'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}
              precision    recall  f1-score   support

           0       0.56      0.52      0.54       143
           1       0.58      0.62      0.60       151

    accuracy                           0.57       294
   macro avg       0.57      0.57      0.57       294
weighted avg       0.57      0.57      0.57       294

RF ROC-AUC: 0.6127

--- Training XGBoost (Tuned) ---


Best XGB Params: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200, 'subsample': 1.0}
              precision    recall  f1-score   support

           0       0.58      0.63      0.61       143
           1       0.62      0.58      0.60       151

    accuracy                           0.60       294
   macro avg       0.60      0.60      0.60       294
weighted avg       0.60      0.60      0.60       294

XGB ROC-AUC: 0.6521

Selecting XGBoost as the final model.
Saved 'best_model.pkl'.
